## Mô Hình M3: Transfer Learning Pretrained ImageNet

Báo cáo kiến trúc, chiến lược huấn luyện 2 pha (Two-Phase Fine-Tuning) và đánh giá họ mô hình chuyển giao tri thức **M3Transfer** trên tập dữ liệu GTSRB.

**Mục tiêu chính**:
1. Đánh giá 3 trường phái kiến trúc kinh điển được huấn luyện trước trên 1.28 triệu ảnh ImageNet:
   * **ResNet18**: Kiến trúc tích chập dày đặc truyền thống kết hợp khối BasicBlock.
   * **MobileNetV2**: Tích chập tách biệt theo chiều sâu (Depthwise Separable Conv), Inverted Residual và Linear Bottleneck.
   * **EfficientNet-B0**: Định cỡ đồng nhất (Compound Scaling) kết hợp khối MBConv và cơ chế chú ý kênh Squeeze-and-Excitation (SE).
2. Giải quyết bài toán không tương thích độ phân giải: Cơ chế nội suy song tuyến (Bilinear Interpolation) từ $48 \times 48 \rightarrow 224 \times 224$ để phù hợp hệ số giảm mẫu $32\times$ của các backbone pretrained.
3. Áp dụng kỹ thuật huấn luyện 2 pha (Two-Phase Training) ngăn ngừa hiện tượng quên thảm khốc (Catastrophic Forgetting).

## 1. Khởi tạo Môi trường và Lựa chọn Backbone

### 1.1. Khởi tạo thư viện và cấu hình thư mục làm việc

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, precision_recall_fscore_support

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision
from torchvision.transforms import v2

# Thiết lập thư mục gốc của dự án
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
print(f"Thư mục làm việc: {Path.cwd()}")

Thư mục làm việc: d:\code\Python\DSEB_ex\DL\gtsrb-model-comparison


### 1.2. Lựa chọn Backbone và Cờ điều khiển
* `BACKBONE`: Chọn giữa 3 tùy chọn: `"resnet18"`, `"mobilenetv2"`, `"effnetb0"`.
* `FORCE_RETRAIN = False`: Mặc định nạp checkpoint tương ứng đã huấn luyện trong thư mục `checkpoints/`.

In [2]:
BACKBONE = "resnet18"       # Tuỳ chọn: "resnet18" | "mobilenetv2" | "effnetb0"
FORCE_RETRAIN = False      # Đặt True nếu muốn huấn luyện lại từ đầu
DEMO_EPOCHS = 2           # Số epoch chạy demo
PHASE1_EPOCHS = 3         # Epoch đóng băng backbone
PHASE2_EPOCHS = 15        # Epoch mở băng fine-tune
BATCH_SIZE = 64
IMG_SIZE = 224            # Độ phân giải bắt buộc cho pretrained backbone

INDEX_CSV = Path("data/processed/index.csv")
CKPT_PATH = Path(f"checkpoints/m3_{BACKBONE}_best.pt")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Backbone đã chọn: {BACKBONE}")
print(f"Thiết bị tính toán: {DEVICE}")
print(f"Đường dẫn Checkpoint: {CKPT_PATH}")

Backbone đã chọn: resnet18
Thiết bị tính toán: cpu
Đường dẫn Checkpoint: checkpoints\m3_resnet18_best.pt


## 2. Pipeline Dữ Liệu 224x224 & Chuẩn Hóa ImageNet

### 2.1. Cơ sở lý thuyết về Scale và Phân phối Chuẩn Hóa
1. **Vì sao phải upsample $48 \times 48 \rightarrow 224 \times 224$?**
   * Các backbone ResNet18, MobileNetV2, EfficientNet-B0 đều có cấu trúc giảm mẫu không gian tổng cộng 32 lần (5 lần stride 2 hoặc MaxPool).
   * Nếu đưa ảnh kích thước gốc $48 \times 48$ vào: Feature map cuối cùng chỉ còn $48 / 32 = 1.5 \rightarrow$ làm tròn thành $2 \times 2$, làm mất hoàn toàn cấu trúc không gian và khiến các tầng sâu trở nên vô dụng.
   * Độ phân giải $224 \times 224$ cho kích thước feature map cuối cùng $224 / 32 = 7 \times 7$, khớp chính xác với tỉ lệ (scale và stride) mà các trọng số pretrained được tối ưu.
2. **Chuẩn hóa ImageNet Statistics**:
   * Sử dụng `mean=(0.485, 0.456, 0.406)` và `std=(0.229, 0.224, 0.225)` của ImageNet thay vì GTSRB. Điều này bắt buộc vì các bộ lọc trích xuất đặc trưng tầng đầu và các tham số `running_mean`/`running_var` trong các lớp BatchNorm pretrained được tích lũy trên phân phối này.

In [3]:
NUM_CLASSES = 43
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def _build_normalize_imagenet() -> v2.Transform:
    return v2.Compose([
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=list(IMAGENET_MEAN), std=list(IMAGENET_STD)),
    ])

def build_train_transform(img_size: int = 224) -> v2.Transform:
    return v2.Compose([
        v2.RandomAffine(degrees=15.0, translate=(0.10, 0.10), scale=(0.9, 1.1),
                        interpolation=v2.InterpolationMode.BILINEAR, fill=0),
        v2.ColorJitter(brightness=0.20, contrast=0.20),
    ])

### 2.2. Lớp Dữ Liệu Tích Hợp Nội Suy Song Tuyến (`GTSRBDataset`)
Nạp ảnh $48 \times 48$ từ cache và tự động nội suy song tuyến (Bilinear Interpolation) lên $224 \times 224$ trong hàm `__getitem__`.

In [4]:
class GTSRBDataset224(Dataset):
    def __init__(self, index_csv: str | Path, split: str, img_size: int = 224,
                 preprocess: str = "clahe", processed_dir: str | Path = "data/processed",
                 cache_size: int = 48, transform=None) -> None:
        self.split = split
        self.img_size = img_size
        self.transform = transform

        frame = pd.read_csv(index_csv)
        self.frame = frame[frame["split"] == split].reset_index(drop=True)
        self._cache_rows = frame.index[frame["split"] == split].to_numpy()
        self.normalizer = _build_normalize_imagenet()

        path = Path(processed_dir) / f"images_{cache_size}_{preprocess}.npy"
        self.cache = np.load(path, mmap_mode="r")

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, i: int):
        image = np.array(self.cache[self._cache_rows[i]], copy=True)
        label = int(self.frame.iloc[i]["class_id"])
        tensor = torch.from_numpy(np.ascontiguousarray(image)).permute(2, 0, 1)  # CHW

        if self.transform is not None:
            tensor = self.transform(tensor)

        # Nội suy song tuyến từ 48x48 lên 224x224
        if tensor.shape[-1] != self.img_size:
            tensor = F.interpolate(
                tensor.unsqueeze(0).float(),
                size=(self.img_size, self.img_size),
                mode="bilinear", align_corners=False,
            ).squeeze(0).clamp(0, 255).to(torch.uint8)

        return self.normalizer(tensor), label

def make_loader_224(index_csv: str | Path, split: str, batch_size: int = 64,
                    transform=None, num_workers: int = 0) -> DataLoader:
    ds = GTSRBDataset224(index_csv, split, img_size=224, transform=transform)
    is_train = split == "train"
    return DataLoader(ds, batch_size=batch_size, shuffle=is_train, drop_last=is_train, num_workers=num_workers)

train_tf = build_train_transform(224)
train_loader = make_loader_224(INDEX_CSV, "train", batch_size=BATCH_SIZE, transform=train_tf)
val_loader = make_loader_224(INDEX_CSV, "val", batch_size=BATCH_SIZE)
test_loader = make_loader_224(INDEX_CSV, "test", batch_size=BATCH_SIZE)

print(f"DataLoaders 224x224: Train={len(train_loader.dataset)}, Val={len(val_loader.dataset)}, Test={len(test_loader.dataset)} mẫu.")

DataLoaders 224x224: Train=31379, Val=7830, Test=12630 mẫu.


## 3. Xây Dựng Kiến Trúc `M3Transfer`

### 3.1. Cơ chế thay thế Classifier Head 43 lớp
Nạp mô hình gốc từ `torchvision.models` và thay thế lớp phân loại 1.000 lớp ImageNet bằng lớp phân loại 43 lớp biển báo:
* **ResNet18**: Thay `net.fc = nn.Linear(512, 43)`
* **MobileNetV2**: Thay `net.classifier = Sequential(Dropout(0.2), nn.Linear(1280, 43))`
* **EfficientNet-B0**: Thay `net.classifier = Sequential(Dropout(0.2), nn.Linear(1280, 43))`

In [5]:
BACKBONES = {
    "resnet18": ("resnet18", "ResNet18_Weights"),
    "mobilenetv2": ("mobilenet_v2", "MobileNet_V2_Weights"),
    "effnetb0": ("efficientnet_b0", "EfficientNet_B0_Weights"),
}

class M3Transfer(nn.Module):
    """Lớp bọc backbone pretrained ImageNet và thay thế classification head."""
    def __init__(self, backbone: str = "resnet18", num_classes: int = NUM_CLASSES,
                 pretrained: bool = True, head_dropout: float = 0.2) -> None:
        super().__init__()
        if backbone not in BACKBONES:
            raise ValueError(f"Backbone phải thuộc {list(BACKBONES.keys())}, nhận được {backbone}")
        self.backbone_name = backbone
        builder_name, weights_enum_name = BACKBONES[backbone]

        builder = getattr(torchvision.models, builder_name)
        weights = getattr(torchvision.models, weights_enum_name).IMAGENET1K_V1 if pretrained else None
        self.net = builder(weights=weights)

        # Thay thế Classifier Head
        if backbone == "resnet18":
            in_features = self.net.fc.in_features
            self.net.fc = nn.Linear(in_features, num_classes)
        else:
            in_features = self.net.classifier[-1].in_features
            self.net.classifier = nn.Sequential(
                nn.Dropout(head_dropout),
                nn.Linear(in_features, num_classes)
            )
        self.in_features = in_features

    def set_backbone_frozen(self, frozen: bool) -> None:
        """Đóng băng (Freeze) hoặc Mở băng (Unfreeze) toàn bộ trọng số của backbone."""
        head_module = self.net.fc if self.backbone_name == "resnet18" else self.net.classifier
        head_params = set(head_module.parameters())
        for p in self.net.parameters():
            if p not in head_params:
                p.requires_grad = not frozen

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

### 3.2. Khởi tạo mô hình và kiểm tra số lượng tham số

In [6]:
model_demo = M3Transfer(backbone=BACKBONE, pretrained=True)
total_params = sum(p.numel() for p in model_demo.parameters())
trainable_params = sum(p.numel() for p in model_demo.parameters() if p.requires_grad)

print(f"Mô hình M3 ({BACKBONE}):")
print(f"  Tổng số tham số      : {total_params:,}")
print(f"  Tham số có thể train : {trainable_params:,}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\admin/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100.0%


Mô hình M3 (resnet18):
  Tổng số tham số      : 11,198,571
  Tham số có thể train : 11,198,571


## 4. Chiến Lược Huấn Luyện Hai Pha (Two-Phase Training)

### 4.1. Phân tích Hiện tượng Catastrophic Forgetting
* **Pha 1 (Epoch 1 - 3): Freeze Backbone**: Lớp Head mới được khởi tạo ngẫu nhiên nên loss ban đầu rất lớn ($\approx \ln 43 \approx 3.76$). Nếu mở băng toàn bộ mạng ngay từ đầu, gradient cực lớn này sẽ chảy ngược và phá hủy toàn bộ các trọng số tiền huấn luyện ImageNet quý giá.
* **Pha 2 (Epoch 4+): Unfreeze Fine-Tuning**: Mở băng toàn bộ mạng, sử dụng thuật toán tối ưu `AdamW` với learning rate rất nhỏ ($10^{-4}$ đến $10^{-5}$) để tinh chỉnh nhẹ nhàng các tầng sâu mà không làm mất tri thức khái quát.

In [7]:
@torch.no_grad()
def predict(model: nn.Module, loader: DataLoader, device: torch.device) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    probs, targets = [], []
    for images, y in loader:
        logits = model(images.to(device))
        probs.append(torch.softmax(logits.float(), dim=1).cpu())
        targets.append(y)
    return torch.cat(targets).numpy(), torch.cat(probs).numpy()

def compute_metrics(y_true: np.ndarray, y_prob: np.ndarray) -> dict:
    y_pred = y_prob.argmax(axis=1)
    top1 = float((y_pred == y_true).mean())
    top5_idx = np.argsort(-y_prob, axis=1)[:, :5]
    top5 = float((top5_idx == y_true[:, None]).any(axis=1).mean())
    macro_f1 = float(f1_score(y_true, y_pred, labels=list(range(NUM_CLASSES)), average="macro", zero_division=0))
    return {"top1": top1, "top5": top5, "macro_f1": macro_f1, "y_true": y_true, "y_pred": y_pred}

def evaluate(model: nn.Module, loader: DataLoader, device: torch.device) -> dict:
    y_true, y_prob = predict(model, loader, device)
    return compute_metrics(y_true, y_prob)

def train_epoch(model: nn.Module, loader: DataLoader, criterion: nn.Module,
                optimizer: torch.optim.Optimizer, device: torch.device) -> float:
    model.train()
    total_loss = 0.0
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(targets)
    return total_loss / len(loader.dataset)

def train_m3(backbone: str = BACKBONE, p1_epochs: int = 3, p2_epochs: int = 15, save_path: Path = CKPT_PATH) -> M3Transfer:
    model = M3Transfer(backbone=backbone, pretrained=True).to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    best_f1 = 0.0
    save_path.parent.mkdir(parents=True, exist_ok=True)

    # --- Pha 1: Đóng băng Backbone, chỉ train Head ---
    print(f"=== PHA 1: FREEZE BACKBONE ({p1_epochs} epoch, lr=1e-3) ===")
    model.set_backbone_frozen(True)
    opt_head = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
    for ep in range(1, p1_epochs + 1):
        loss = train_epoch(model, train_loader, criterion, opt_head, DEVICE)
        val_m = evaluate(model, val_loader, DEVICE)
        print(f"Pha 1 - Epoch {ep}/{p1_epochs} | Loss: {loss:.4f} | Val Top-1: {val_m['top1']*100:.2f}% | Val Macro-F1: {val_m['macro_f1']:.4f}")

    # --- Pha 2: Mở băng toàn mạng, fine-tune với learning rate nhỏ ---
    print(f"\n=== PHA 2: UNFREEZE FINE-TUNING ({p2_epochs} epoch, lr=1e-4) ===")
    model.set_backbone_frozen(False)
    opt_all = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt_all, T_max=p2_epochs, eta_min=1e-6)
    for ep in range(1, p2_epochs + 1):
        loss = train_epoch(model, train_loader, criterion, opt_all, DEVICE)
        val_m = evaluate(model, val_loader, DEVICE)
        sched.step()

        is_best = val_m["macro_f1"] > best_f1
        if is_best:
            best_f1 = val_m["macro_f1"]
            torch.save(model.state_dict(), save_path)
        tag = "* [BEST]" if is_best else ""
        print(f"Pha 2 - Epoch {ep:02d}/{p2_epochs:02d} | Loss: {loss:.4f} | Val Top-1: {val_m['top1']*100:.2f}% | Val Macro-F1: {val_m['macro_f1']:.4f} {tag}")

    print(f"\nHuấn luyện hoàn tất. Checkpoint tốt nhất lưu tại: {save_path}")
    ckpt = torch.load(save_path, map_location=DEVICE, weights_only=False)
    state = ckpt["model_state"] if isinstance(ckpt, dict) and "model_state" in ckpt else ckpt
    model.load_state_dict(state)
    return model

## 5. Thực Thi: Nạp Checkpoint Hoặc Huấn Luyện Mới

In [8]:
model = M3Transfer(backbone=BACKBONE, pretrained=False).to(DEVICE)

if not FORCE_RETRAIN and CKPT_PATH.exists():
    print(f"[Tài nguyên sẵn sàng] Nạp checkpoint mô hình từ {CKPT_PATH}")
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
    state_dict = ckpt["model_state"] if isinstance(ckpt, dict) and "model_state" in ckpt else ckpt
    model.load_state_dict(state_dict)
    print("Nạp trọng số thành công.")
else:
    p1 = 1 if not FORCE_RETRAIN else PHASE1_EPOCHS
    p2 = DEMO_EPOCHS if not FORCE_RETRAIN else PHASE2_EPOCHS
    print(f"[Thực thi] Bắt đầu huấn luyện Transfer Learning ({p1} ep pha 1 + {p2} ep pha 2)...\n")
    model = train_m3(backbone=BACKBONE, p1_epochs=p1, p2_epochs=p2)

[Tài nguyên sẵn sàng] Nạp checkpoint mô hình từ checkpoints\m3_resnet18_best.pt
Nạp trọng số thành công.


## 6. Báo Cáo Kết Quả Đánh Giá Trên Tập Test Chính Thức

In [9]:
test_metrics = evaluate(model, test_loader, DEVICE)

print("=" * 50)
print(f"KẾT QUẢ KIỂM THỬ MÔ HÌNH M3 ({BACKBONE.upper()})")
print("=" * 50)
print(f"Top-1 Accuracy : {test_metrics['top1'] * 100:.2f}%")
print(f"Top-5 Accuracy : {test_metrics['top5'] * 100:.2f}%")
print(f"Macro-F1       : {test_metrics['macro_f1']:.4f}")
print("=" * 50)

KẾT QUẢ KIỂM THỬ MÔ HÌNH M3 (RESNET18)
Top-1 Accuracy : 99.32%
Top-5 Accuracy : 99.94%
Macro-F1       : 0.9903
